In [1]:
from deep_kolmogorov import modeling, pdes, trainer
import torch
import numpy as np

In [281]:
import importlib
importlib.reload(trainer)
pde = pdes.BSasian()
import numpy as np
K = 100
num_path = 10000
device = torch.device("cuda")
config = {"s":10, "r":0.02, "sigma": 0.3, "kappa": 1.01}

import importlib
importlib.reload(pdes)
pde = pdes.BSasian()

with torch.no_grad(): 
    data = {
        key: torch.full(((K+1)*num_path, 1), value, dtype=torch.float32, device=device) for key, value in config.items()
    }
    data.update(
        {
            "t": torch.full((K + 1,), 0.6, dtype=torch.float32, device=device).repeat(num_path).reshape(-1,1)
        }
    )

data["x"] = torch.concat([data["s"], data["s"]], dim=1)
data["K"] = pdes.BSasian.get_K(data)
payoff = pde.sde(data)

C0 = torch.mean(payoff)
cstd = torch.std(payoff)
print("Estimated price: ", C0)
print("Estimated CI: ", f"[{C0 - 1.96*cstd/np.sqrt(K*num_path)}, {C0 + 1.96*cstd/np.sqrt(K*num_path)}]")

print("exact price: ", (torch.exp(-data["t"]*data["r"]) * pde.solution(data))[0,0])

n is 250
Estimated price:  tensor(0.1235, device='cuda:0')
Estimated CI:  [0.12306501716375351, 0.12392885237932205]
exact price:  tensor(0.1234, device='cuda:0')


In [133]:
0.001/0.067

0.014925373134328358

In [126]:
d1

-0.03299873109647069

In [127]:
d2

-0.03317193617722758

In [33]:
import importlib
importlib.reload(trainer)
pde = pdes.BSTI()
import numpy as np
K = 100
num_path = 10000
device = torch.device("cuda")
config = {"s":10, "r0":0.02, "r1":2, "r2":2, "sigma_bar": 0.3, "beta":0.02, "kappa": 1.01}

import importlib
importlib.reload(pdes)
pde = pdes.BSTI()

with torch.no_grad(): 
    data = {
        key: torch.full(((K+1)*num_path, 1), value, dtype=torch.float32, device=device) for key, value in config.items()
    }
    data.update(
        {
            "t": torch.full((K + 1,), 0.1, dtype=torch.float32, device=device).repeat(num_path).reshape(-1,1)
        }
    )

data["x"] = data["s"].clone()
data["K"] = pdes.BSTI.get_K(data)
data["r"] = pdes.BSTI.get_r(data)
data["sigma"] = pdes.BSTI.get_sigma(data)
payoff = pde.sde(data)

C0 = torch.mean(payoff)
cstd = torch.std(payoff)
print("Estimated price: ", C0)
print("Estimated CI: ", f"[{C0 - 1.96*cstd/np.sqrt(K*num_path)}, {C0 + 1.96*cstd/np.sqrt(K*num_path)}]")

print("exact price: ", (torch.exp(-pde.get_rmt(0, data["t"], data["r0"], data["r1"], data["r2"])) * pde.solution(data))[0,0])

Estimated price:  tensor(8.0087, device='cuda:0')
Estimated CI:  [8.003179550170898, 8.014310836791992]
exact price:  tensor(8.0043, device='cuda:0')


In [208]:
0.01/0.74

0.008620689655172415

In [143]:
pde.get_rmt(0, data["t"], data["r0"], data["r1"], data["r2"])[0,0]

tensor(99.9000, device='cuda:0')

In [41]:
import importlib
importlib.reload(trainer)
import numpy as np
K = 100
num_path = 10000
device = torch.device("cuda")
config = {"s":10, "r0":0.02, "r1":0.002, "r2":0.02, "sigma_bar": 0.3, "beta":0.02, "rho": 0.1, "kappa": 1.01}

import importlib
importlib.reload(pdes)
pde = pdes.BSbasketTI()

with torch.no_grad(): 
    data = {
        key: torch.full(((K+1)*num_path,  pde.hypercubes[key].dims[0]), value, dtype=torch.float32, device=device) for key, value in config.items()
    }
    data.update(
        {
            "t": torch.full((K + 1,), 0.1, dtype=torch.float32, device=device).repeat(num_path).reshape(-1,1)
        }
    )

data["x"] = data["s"].clone()
data["K"] = pdes.BSbasketTI.get_K(data)
data["r"] = pdes.BSbasketTI.get_r(data)
# data["sigma"] = pdes.BSbasketTI.get_sigma(data)
payoff = pde.sde(data)

C0 = torch.mean(payoff)
cstd = torch.std(payoff)
print("Estimated price: ", C0)
print("Estimated CI: ", f"[{C0 - 1.96*cstd/np.sqrt(K*num_path)}, {C0 + 1.96*cstd/np.sqrt(K*num_path)}]")

print("exact price: ", (torch.exp(-pde.get_rmt(0, data["t"], data["r0"], data["r1"], data["r2"])) * pde.solution(data))[0,0])

Estimated price:  tensor(0.4012, device='cuda:0')
Estimated CI:  [0.39986756443977356, 0.40257909893989563]
exact price:  tensor(0.4014, device='cuda:0')
